In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install gymnasium
!pip install stable-baselines3

In [ ]:
import gymnasium as gym

from gymnasium import spaces

import numpy as np

import matplotlib.pyplot as plt

from stable_baselines3 import PPO

In [ ]:
class MultiAgentSaudiEnv(gym.Env):

    def __init__(self, num_agents=3):

        super(MultiAgentSaudiEnv, self).__init__()

        self.num_agents = num_agents

        # =====================================
        # LOAD SAUDI TENSOR
        # =====================================

        tensor_path = (
            "/content/drive/MyDrive/"
            "PyroRL_Saudi_Project/datasets/"
            "saudi_eastern_province/grids/32x32/"
            "state_tensor.npy"
        )

        self.initial_state = np.load(tensor_path)

        self.state = self.initial_state.copy()

        # =====================================
        # OBSERVATION SPACE
        # =====================================

        self.observation_space = spaces.Box(
            low=0,
            high=1,
            shape=(7,32,32),
            dtype=np.float32
        )

        # =====================================
        # MULTI-AGENT ACTION SPACE
        # =====================================

        self.action_space = spaces.MultiDiscrete(
            [5] * self.num_agents
        )

        # =====================================
        # AGENT POSITIONS
        # =====================================

        self.agent_positions = []

        for idx in range(self.num_agents):

            x = np.random.randint(0, 32)

            y = np.random.randint(0, 32)

            self.agent_positions.append([x, y])

        self.current_step = 0

        self.max_steps = 200

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.state = self.initial_state.copy()

        self.agent_positions = []

        for idx in range(self.num_agents):

            x = np.random.randint(0, 32)

            y = np.random.randint(0, 32)

            self.agent_positions.append([x, y])

        self.current_step = 0

        return self.state.astype(np.float32), {}

    def step(self, actions):

        fire_layer = self.state[0]

        fuel_layer = self.state[1]

        wind_x = self.state[2]

        wind_y = self.state[3]

        terrain_layer = self.state[4]

        # =====================================
        # MULTI-AGENT MOVEMENT
        # =====================================

        for idx, action in enumerate(actions):

            x, y = self.agent_positions[idx]

            if action == 0:
                x -= 1

            elif action == 1:
                x += 1

            elif action == 2:
                y -= 1

            elif action == 3:
                y += 1

            x = np.clip(x, 0, 31)

            y = np.clip(y, 0, 31)

            self.agent_positions[idx] = [x, y]

        # =====================================
        # SPATIAL FIRE SPREAD
        # =====================================

        new_fire_layer = fire_layer.copy()

        for i in range(1,31):

            for j in range(1,31):

                current_fire = fire_layer[i,j]

                if current_fire < 0.05:
                    continue

                neighbors = [

                    (i-1,j),

                    (i+1,j),

                    (i,j-1),

                    (i,j+1)
                ]

                for ni, nj in neighbors:

                    fuel = fuel_layer[ni,nj]

                    wind_bonus = (

                        abs(wind_x[ni,nj])

                        +

                        abs(wind_y[ni,nj])
                    )

                    terrain_factor = (

                        1 + terrain_layer[ni,nj]
                    )

                    spread_amount = (

                        0.02
                        * current_fire
                        * fuel
                        * (1 + wind_bonus)
                        * terrain_factor
                    )

                    new_fire_layer[ni,nj] += spread_amount

        fire_layer = np.clip(
            new_fire_layer,
            0,
            1
        )

        # =====================================
        # MULTI-AGENT SUPPRESSION
        # =====================================

        suppression_power = 0.05

        for x, y in self.agent_positions:

            fire_layer[x,y] = max(

                0,

                fire_layer[x,y]
                - suppression_power
            )
        # =====================================
        # FUEL CONSUMPTION
        # =====================================

        fuel_layer = np.clip(

            fuel_layer
            - 0.01 * fire_layer,

            0,
            1
        )

        self.state[1] = fuel_layer

        # =====================================
        # COOPERATIVE REWARD
        # =====================================

        mean_fire = np.mean(fire_layer)

        reward = -mean_fire

        # Bonus for containment

        if mean_fire < 0.2:

            reward += 2.0

        self.state[0] = fire_layer

        self.current_step += 1

        done = (
            self.current_step
            >=
            self.max_steps
        )

        truncated = False

        return (

            self.state.astype(np.float32),

            reward,

            done,

            truncated,

            {}
        )

In [ ]:
env = MultiAgentSaudiEnv(
    num_agents=3
)

print("Multi-agent environment created.")

In [ ]:
env = MultiAgentSaudiEnv(
    num_agents=3
)

obs, info = env.reset()

print(obs.shape)

In [ ]:
actions = [0, 1, 4]

obs, reward, done, truncated, info = env.step(actions)

print("Reward:", reward)

print("Agent Positions:")

print(env.agent_positions)

In [ ]:
import torch as th
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with th.no_grad():
            n_flatten = self.cnn(
                th.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: th.Tensor) -> th.Tensor:
        return self.linear(self.cnn(observations))

policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=256),
    normalize_images=False,
)

model = PPO(
    "CnnPolicy",
    env,
    verbose=1,
    policy_kwargs=policy_kwargs
)

model.learn(
    total_timesteps=20000
)

print("Multi-agent PPO training completed.")

In [ ]:
obs, info = env.reset()

fire_values = []

for _ in range(100):

    action, _ = model.predict(obs)

    obs, reward, done, truncated, info = env.step(action)

    fire_values.append(
        np.mean(env.state[0])
    )

    if done:
        break

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(fire_values)

plt.title(
    "Multi-Agent Wildfire Containment"
)

plt.xlabel("Step")

plt.ylabel("Mean Fire Intensity")

plt.grid()

plt.show()

In [ ]:
fire_layer = env.state[0]

plt.figure(figsize=(7,7))

plt.imshow(fire_layer)

for idx, (x,y) in enumerate(
    env.agent_positions
):

    plt.scatter(

        y,

        x,

        s=200,

        marker='X',

        label=f'Agent {idx+1}'
    )

plt.legend()

plt.title(
    "Multi-Agent Fire Suppression"
)

plt.colorbar()

plt.show()